## VirtualiZarr + GOES-West: Full Disk False-Green Composite

Build xarray Dataset out of GOES-18 CONUS NetCDF files from S3 and render a false-green RGB composite with gamma correction

In [ ]:
import re
import warnings
from datetime import UTC, datetime, timedelta, timezone

import matplotlib.pyplot as plt
import numpy as np
import obstore as obs
import virtualizarr as vz
import xarray as xr
from box import Box
from obstore.store import S3Store, from_url
from virtualizarr.manifests import ManifestStore
from virtualizarr.parsers import HDFParser
from virtualizarr.registry import ObjectStoreRegistry

warnings.filterwarnings(
    "ignore", message="Numcodecs codecs are not in the Zarr version 3"
)

### params

In [ ]:
params = Box(
    {
        "toi": datetime.now(UTC).date() - timedelta(days=1),  # yesterday
        "bucket": "noaa-goes18",
        "product": "ABI-L2-MCMIPC",
        "region": "us-east-1",
        "bands": ["CMI_C01", "CMI_C02", "CMI_C03"],
    }
)

### TOI (Time of Interest)

Yesterday 10:00-11:00 AM Pacific (PDT) converted to UTC for S3 path construction.

GOES S3 layout: `s3://noaa-goes18/ABI-L2-MCMIPC/{year}/{doy:03d}/{hour:02d}/`

In [ ]:
PDT = timezone(timedelta(hours=-7))
start = datetime(params.toi.year, params.toi.month, params.toi.day, 10, 0, tzinfo=PDT)
end = datetime(params.toi.year, params.toi.month, params.toi.day, 11, 0, tzinfo=PDT)

start_utc = start.astimezone(UTC)
end_utc = end.astimezone(UTC)

print(f"Local: {start:%Y-%m-%d %H:%M} to {end:%H:%M} PDT")
print(f"UTC: {start_utc:%Y-%m-%d %H:%M} to {end_utc:%H:%M} UTC")
print(f"DOY: {start_utc.timetuple().tm_yday:03d}")
print(f"Hours: {start_utc.hour:02d}-{end_utc.hour:02d}")

### Search

Use obstore to list NetCDF files in the public `noaa-goes18` bucket

In [ ]:
store = S3Store.from_url(
    f"s3://{params.bucket}", region=params.region, skip_signature=True
)

#
files = []
hour = start_utc.replace(minute=0, second=0, microsecond=0)
while hour <= end_utc:
    doy = hour.timetuple().tm_yday
    prefix = f"{params.product}/{hour.year}/{doy:03d}/{hour.hour:02d}/"
    result = obs.list_with_delimiter(store, prefix=prefix)
    for obj in result["objects"]:
        path = obj["path"]
        if path.endswith(".nc"):
            #
            m = re.search(r"s(\d{11})", path)
            if m:
                ts = m.group(1)
                yr, dy, hh, mm = int(ts[:4]), int(ts[4:7]), int(ts[7:9]), int(ts[9:11])
                scan_time = datetime(yr, 1, 1, tzinfo=UTC) + timedelta(
                    days=dy - 1, hours=hh, minutes=mm
                )
                if start_utc <= scan_time <= end_utc:
                    files.append(f"s3://{params.bucket}/{path}")
    hour += timedelta(hours=1)

files.sort()
print(f"Found {len(files)} files\n")
for f in files:
    print(f"  {f.split('/')[-1]}")

### VirtualiZarr

references all files, dropping per-band statistics and bands 4-16 (only need C01/C02/C03 for false green)

In [ ]:
def per_band(prefix):
    """per-band variable names for all ABI bands"""
    return [f"{prefix}_C{i:02}" for i in range(1, 17)]


#
drop_variables = (
    per_band("band_id")
    + per_band("min_reflectance_factor")
    + per_band("max_reflectance_factor")
    + per_band("mean_reflectance_factor")
    + per_band("std_dev_reflectance_factor")
    + per_band("min_brightness_temperature")
    + per_band("max_brightness_temperature")
    + per_band("mean_brightness_temperature")
    + per_band("std_dev_brightness_temperature")
    + per_band("outlier_pixel_count")
    + [f"CMI_C{i:02}" for i in range(4, 17)]
    + [f"DQF_C{i:02}" for i in range(1, 17)]
)

#
loadable_vars = [
    "x",
    "y",
    "t",
    "goes_imager_projection",
    "x_image_bounds",
    "y_image_bounds",
    "nominal_satellite_subpoint_lat",
    "nominal_satellite_subpoint_lon",
    "nominal_satellite_height",
    "geospatial_lat_lon_extent",
] + per_band("band_wavelength")

In [ ]:
bucket_url = f"s3://{params.bucket}"
base_store = from_url(bucket_url, region=params.region, skip_signature=True)
registry = ObjectStoreRegistry({bucket_url: base_store})

parser = HDFParser(drop_variables=drop_variables)

vds = vz.open_virtual_mfdataset(
    files,
    registry=registry,
    parser=parser,
    loadable_variables=loadable_vars,
    combine="nested",
    concat_dim="time",
    coords="minimal",
    compat="override",
)

# TODO: slow

In [ ]:
vds.vz.nbytes, vds.nbytes

In [ ]:
vds

### ManifestStore

ManifestStore wraps the virtual references as a Zarr store

In [ ]:
from virtualizarr.manifests import ManifestArray, ManifestGroup
from virtualizarr.manifests.utils import copy_and_replace_metadata

#
virtual_arrays = {}
for name, var in vds.variables.items():
    if isinstance(var.data, ManifestArray):
        marr = var.data
        #
        restored_metadata = copy_and_replace_metadata(
            marr.metadata, new_dimension_names=var.dims, new_attributes=dict(var.attrs)
        )
        virtual_arrays[name] = ManifestArray(
            chunkmanifest=marr.manifest, metadata=restored_metadata
        )

group = ManifestGroup(arrays=virtual_arrays, attributes=dict(vds.attrs))
manifest_store = ManifestStore(group, registry=registry)
ds = xr.open_zarr(manifest_store, zarr_format=3, consolidated=False)

#
loadable = {
    name: var
    for name, var in vds.variables.items()
    if not isinstance(var.data, ManifestArray)
}
ds = ds.assign(loadable)

In [ ]:
ds

### CONUS

first timestep and false-green bands

In [ ]:
frame = ds[params.bands].isel(time=0)
dict(frame.dims)

### False-green RGB

In [ ]:
c01 = frame.CMI_C01.values
c02 = frame.CMI_C02.values
c03 = frame.CMI_C03.values

r = c02
g = 0.45 * c02 + 0.10 * c03 + 0.45 * c01  # a false green formula
b = c01

rgb = np.stack([r, g, b], axis=-1)
rgb = np.clip(rgb, 0, 1)  # reflectances between 0 and 1
rgb.shape, rgb.min(), rgb.max()

### Gamma

In [ ]:
GAMMA = 2.2
rgb_gamma = np.power(rgb, 1.0 / GAMMA)

fig, axes = plt.subplots(1, 2, figsize=(18, 8))

axes[0].imshow(rgb)
axes[0].set_title("Raw false-green")
axes[0].axis("off")

axes[1].imshow(rgb_gamma)
axes[1].set_title(f"Gamma corrected (1/{GAMMA})")
axes[1].axis("off")

fig.suptitle(
    f"GOES-18 False Green — CONUS — {start:%Y-%m-%d %H:%M} PDT",
    fontsize=14,
    y=1.02,
)
plt.tight_layout()
plt.show()